<a href="https://colab.research.google.com/github/Hubery-1003/Lunar-Landing/blob/main/%E3%80%8CDeep_Q_Learning_for_Lunar_Landing_Partial_Code%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning for Lunar Landing

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!apt-get install -y swig
!pip install gymnasium[box2d]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.1 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 49 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 0s (3,295 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 123632 files and directories currently installed.)
Preparing to unpack .../swi

### Importing the libraries

In [ ]:
import os
import random
import numpy as np
import torch
#NN is neuron network module for torch library
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.autograd as autograd
from torch.autograd import Variable
from collections import deque, namedtuple

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [ ]:
#生成neuron network class
class Network(nn.Module):

  def __init__(self,state_size,action_size,seed=42):
    super(Network,self).__init__()
    self.seed = torch.manual_seed(seed)

    #self.fc1 = nn.Linear(state_size,64) 第一層基本上第一個參數是輸入層(input layer)的神經元數量，基本上是state_size，第二個參數是全連接層的數量-->64
    #有很多個實驗設法建立一個性能AI，設法正確降落在月球上，第一個全連接層的最佳神經元實際是64個
    #actually,the best architecture contains two intermediate fully connection layers

    self.fc1 = nn.Linear(state_size,64)
    self.fc2 = nn.Linear(64,64)
    self.fc3 = nn.Linear(64,action_size)

  def forward(self,state):
    #實際上self.fc1(state)他會返回一個全連接層
    x = self.fc1(state)
    #import torch.nn.functional as F
    #我們需要用rectifier activation function.這個函式庫的relu方法來更新x值藉此來激活這個signal
    x = F.relu(x)
    #以上兩行程式碼代表signal從input layer 傳遞給 first fully connection layer
    x = self.fc2(x)
    x = F.relu(x)
    #最後返回output layer
    return self.fc3(x)

## Part 2 - Training the AI

### Setting up the environment

In [ ]:
#引入gymnasium程式庫
import gymnasium as gym
#建造要執行的遊戲環境，我們要訓練ai的遊戲環境是LunarLander-v3
env = gym.make('LunarLander-v3') # The Lunar Lander environment was upgraded to v3
#state_shape這邊輸入的是遊戲的八維向量 8-dimensional vector
state_shape = env.observation_space.shape
#實際上輸入層的輸入狀態元素數量意味著描述每個時間的八個參數(八維向量)，
#the number of elements in this input state meaning the eight parameters describing at each time
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape: ', state_shape)
print('State size: ', state_size)
print('Number of actions: ', number_actions)

State shape:  (8,)
State size:  8
Number of actions:  4


### Initializing the hyperparameters


In [ ]:
#學習率
learning_rate = 5e-4 #0.005
#從緩衝區抽取的數值數量
minibatch_size = 100
#折扣因子
discount_factor = 0.99
#緩衝區數值總數
replay_buffer_size = int(1e5)
interpolation_parameter = 1e-3

### Implementing Experience Replay

In [ ]:
class ReplayMemory(object):
  def __init__(self,capacity):
    #使用GPU或者CPU訓練
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    #緩衝區的大小
    self.capacity = capacity
    #儲存數據的地方->list[]
    self.memory = []
  #將訓練的數據存入緩衝區(buffer)包括state,action...等
  def push(self,event):
    self.memory.append(event)
    #確保memorybuffer不會超過capacity(buffer的量)
    if len(self.memory) > self.capacity:
    #del means delete ->超過的話就把舊資料刪除
     del self.memory[0]
  #從緩衝區抽取小量樣本並將其轉換成Pytorch張量，並返回這些張量用作於訓練使用
  def sample(self,batch_size):
    experiences = random.sample(self.memory,k = batch_size)
    #np.vstack=>extract and stack data
    #torch.from_numpy=>convert arrat to Pytorch tensor
    #but we also need to make sure that the data type of these states, now PyTorch tensors is only float,and that's again, => states include angles,velocity(速度),values,coordinate
    states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float().to(self.device)
    # action can be 0,1,2,3 => set datatype equals long(long integer)
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long().to(self.device)
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float().to(self.device)
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float().to(self.device)
    #the astype(np.uint8) is used to represent Boolean values before converting them into float tensors
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float().to(self.device)
    return states, next_states, actions, rewards, dones

### Implementing the DQN class

In [ ]:
class Agent():
  def __init__(self,state_size,action_size):
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    #表示每個環境的state數量
    self.state_size = state_size
    #表示每個環境的action數量
    self.action_size = action_size
    #本地 Q 網絡負責根據當前狀態預測 Q 值。這裡 Network 是一個神經網絡類別，它被實例化為本地 Q 網絡，並移動到相應的設備上（GPU 或 CPU）。
    self.local_qnetwork = Network(state_size,action_size).to(self.device)
    #目標 Q 網絡用來計算 Q 值的目標，幫助穩定訓練過程。它與本地 Q 網絡結構相同，但參數更新頻率較低。
    self.target_qnetwork = Network(state_size,action_size).to(self.device)
    #lr means learning rate
    #使用 Adam 優化器來更新本地 Q 網絡的參數。learning_rate 指定了學習率，控制參數更新的步伐大小。
    self.optimizer = optim.Adam(self.local_qnetwork.parameters(),lr = learning_rate)
    #回放記憶體用來存儲代理人的經驗（狀態、行動、獎勵、新狀態）。replay_buffer_size 指定了回放記憶體的最大容量。
    self.memory = ReplayMemory(replay_buffer_size)
    #time step 初始化時間步驟計數器，用於控制目標網絡的更新頻率等操作。
    self.t_step = 0

  #時間步驟函數
  def step(self,state,action,reward,next_state,done):
    self.memory.push((state,action,reward,next_state,done))
    self.t_step = (self.t_step+1)%4
    if self.t_step == 0 :
      #self.memory是object，而self.memory.memory是這個object的 memory attribute
      if len(self.memory.memory) > minibatch_size:
        experiences = self.memory.sample(100)
        self.learn(experiences,discount_factor)
  #epsilon-greedy action selection policy
  #epsilon = 0. is in order to make it a datatype of float
  #深度強化學習中的關鍵步驟，包括狀態轉換、模型推斷和動作選擇
  def act(self,state,epsilon = 0.):
    #unsqueeze是在這個批次新增一個維度dimension=>有點多維陣列的概念
    # 目的是為了將狀態轉換成具有一個樣本的batch
    state = torch.from_numpy(state).float().unsqueeze(0).to(self.device)
    # 使用評估模式，不會應用dropout和 batch normalization
    self.local_qnetwork.eval()
    #關閉梯度計算，節省內存，加快推斷速度
    with torch.no_grad():
      action_values = self.local_qnetwork(state)
      #恢復訓練模式，在後續訓練中一樣可以使用dropout 和 batch mormalization
      self.local_qnetwork.train()
      #epsilon探索(exploration)與利用(explotitation)比例的參數
      #epsilon通常會隨著時間變小，在前期時多探索=>發現更多可能性及更優的行為策略，後期時較專注於利用已知的最佳策略來達到更號的即時回報
      #當隨機數大於epsilon參數時選擇利用explotitation
      if random.random() > epsilon:
        return np.argmax(action_values.cpu().data.numpy())
      #當隨機數小於或等於epsilon時，選擇探索
      else:
        return random.choice(np.arange(self.action_size))
  def learn(self,experiences,discount_factor):
    #unpack experiences
    states, next_states, actions, rewards, dones = experiences = experiences
    next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)
    #計算目標Q值，不經常更新
    q_targets = rewards + (discount_factor*next_q_targets*(1-dones))
    #計算當前Q值，時常更新
    q_expected = self.local_qnetwork(states).gather(1,actions)
    #計算目標Q值與當前Q值得差值
    loss = F.mse_loss(q_expected,q_targets)
    self.optimizer.zero_grad()
    #將插值反向傳播=>更新self.local_qnetwork
    loss.backward()
    #使用優化器更新參數模型
    self.optimizer.step()

    #在這行程式碼中，使用 self.soft_update 方法對 self.target_qnetwork 進行軟更新。
    #這意味著 self.target_qnetwork 的參數會部分更新為 self.local_qnetwork 的參數，通常是利用一個插值參數 (interpolation_parameter) 進行平滑更新。
    self.soft_update(self.local_qnetwork,self.target_qnetwork,interpolation_parameter)

    #軟更新=>每過幾個時間步驟就更新目標Q網路(根據interpolation_parameter)
  def soft_update(self,local_model,target_model,interpolation_parameter):
    #The Network class inherits from nn.module amd has a paremeter method to retrieve(檢索) Network parameters
    for local_param,target_param in zip(local_model.parameters(),target_model.parameters()):
      #更新target q model 是根據插植法𝜃′=𝜏𝜃 + (1−𝜏)𝜃′(𝜏 = interpolation_parameter)
      target_param.data.copy_(interpolation_parameter*local_param.data+(1.0-interpolation_parameter)*target_param.data)

### Initializing the DQN agent

In [ ]:
agent = Agent(state_size, number_actions)

### Training the DQN agent

In [ ]:
#訓練的資料集數目
number_episodes = 2000
#一個資料集最多執行的timesteps
maximum_number_timesteps_per_episode = 1000
#epsilon參數的變化參數
epsilon_starting_value = 1.0
epsilon_ending_value = 0.01
epsilon_decay_value = 0.995
#epsilon參數
epsilon = epsilon_starting_value
#為甚麼maxlen是100是因為一次抽只抽取minibatch_size數目的experience出來
scores_on_100_episodes = deque(maxlen = 100)

for episode in range(1,number_episodes+1):
  state , _ = env.reset()
  score = 0
  for t in range(maximum_number_timesteps_per_episode):
    action = agent.act(state,epsilon)
    next_state,reward,done,_,_ = env.step(action)
    agent.step(state,action,reward,next_state,done)
    state = next_state
    score += reward
    if done:
      break
  scores_on_100_episodes.append(score)
  epsilon = max(epsilon_ending_value,epsilon_decay_value*epsilon)
  print('\rEpisode {}\tAverage Score:{:.2f}'.format(episode,np.mean(scores_on_100_episodes)),end="")
  if episode % 100 == 0:
    print('\rEpisode {}\tAverage Score:{:.2f}'.format(episode,np.mean(scores_on_100_episodes)))
    #官方說總分超過200就是成功達到目標了
  if np.mean(scores_on_100_episodes) >= 200.0:
    #為甚麼要episode-100，因為其實在episode-100集的時候就達到200分了，只是近100集平均值還沒有達到200，所以要達到近100集平均值到200我們才訓練結束
    print('\nEnvironment solved in {:d} episodes!\tAverage Score:{:.2f}'.format(episode-100,np.mean(scores_on_100_episodes)))
    #訓練完畢，將agent的local_qnetwork狀態存到checkpoint.pth檔案中
    torch.save(agent.local_qnetwork.state_dict(), 'checkpoint.pth')
    break

Episode 35	Average Score:-156.65

KeyboardInterrupt: 

In [ ]:
#test
def evaluate_model(agent, env, num_episodes=10):
  env = gym.make('LunarLander-v3')
  scores=[]
  for episode in range(num_episodes):
    state, _ = env.reset()
    done = False
    score = 0
    while not done:
      action = agent.act(state, epsilon=0.0)
      state, reward, done, _, _ = env.step(action.item())
      score += reward
    scores.append(score)
    print(f"Episode {episode}\tScore: {score}")
  print(f"Average Score over {num_episodes} episodes: {np.mean(scores)}")
  env.close()
agent.local_qnetwork.load_state_dict(torch.load('checkpoint.pth'))
evaluate_model(agent, 'LunarLander-v3')

<ipython-input-29-81b9d6e1fede>:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  agent.local_qnetwork.load_state_dict(torch.load('checkpoint.pth'))


Episode 0	Score: 232.93444034082867
Episode 1	Score: 140.43796484742091
Episode 2	Score: 187.52120028886674
Episode 3	Score: 267.7413618745627
Episode 4	Score: 236.5043170951851
Episode 5	Score: 164.46922350679498
Episode 6	Score: 220.80083421524694
Episode 7	Score: 237.98695949756697
Episode 8	Score: 248.67899210038166
Episode 9	Score: 112.71621320905606
Average Score over 10 episodes: 204.97915069759105


## Part 3 - Visualizing the results

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        #為甚麼沒有看到agent.step()方法，因為我們已經訓練完模型了
        #我們現在只要在inference mode看結果而已
        action = agent.act(state)
        state, reward, done, _, _ = env.step(action.item())
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'LunarLander-v3')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()

In [ ]:
#執行git
!!apt-get install git

['',
 'Reading package lists... 0%',
 '',
 'Reading package lists... 0%',
 '',
 'Reading package lists... 0%',
 '',
 'Reading package lists... 3%',
 '',
 'Reading package lists... 3%',
 '',
 'Reading package lists... 4%',
 '',
 'Reading package lists... 4%',
 '',
 'Reading package lists... 37%',
 '',
 'Reading package lists... 38%',
 '',
 'Reading package lists... 38%',
 '',
 'Reading package lists... 39%',
 '',
 'Reading package lists... 39%',
 '',
 'Reading package lists... 46%',
 '',
 'Reading package lists... 46%',
 '',
 'Reading package lists... 55%',
 '',
 'Reading package lists... 55%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 59%',
 '',
 'Reading package lists... 65%',
 '',
 'Reading package lists... 65%',
 '',
 'Reading pack

In [34]:
!git clone https://github.com/Hubery-1003/Lunar-Landing.git

fatal: destination path 'Lunar-Landing' already exists and is not an empty directory.
